<a href="https://colab.research.google.com/github/fariaesha-prog/AI-Cyberbullying-Detector/blob/main/AI_Cyberbullying_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install pandas scikit-learn gradio

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv("/content/dataset.csv")
df = df[['comment_text', 'toxic']].dropna()

X_train, X_test, y_train, y_test = train_test_split(df['comment_text'], df['toxic'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True,
    stop_words='english',
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced', C=5)
model.fit(X_train_vec, y_train)

print(classification_report(y_test, model.predict(X_test_vec)))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97     28859
           1       0.67      0.84      0.75      3056

    accuracy                           0.95     31915
   macro avg       0.83      0.90      0.86     31915
weighted avg       0.95      0.95      0.95     31915



In [ ]:
import gradio as gr
import pandas as pd

# ── Toxic keyword list ────────────────────────────────────────────────────────
TOXIC_KEYWORDS = {
    "hate","stupid","ugly","idiot","loser","worthless","kill","die",
    "nobody likes you","go away","shut up","dumb","pathetic","disgusting",
    "horrible","terrible","awful","moron","jerk","freak","weirdo",
}

def find_flagged_terms(text):
    found = []
    low = text.lower()
    for kw in TOXIC_KEYWORDS:
        if kw in low:
            found.append(kw)
    return found

session_log = []

def predict(text):
    if not text or not text.strip():
        return (
            gr.update(value="⚠️ Please enter a comment.", visible=True),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), render_history(), render_stats(),
        )

    vec  = vectorizer.transform([text])
    prob = model.predict_proba(vec)[0][1]
    toxic = prob > 0.5
    confidence = round(prob * 100, 1) if toxic else round((1 - prob) * 100, 1)

    flagged = find_flagged_terms(text)
    words   = set(text.lower().split())

    harassment  = min(100, round(prob * 80  + (10 if any(w in words for w in {"nobody","loser","freak"}) else 0), 1))
    hate_speech = min(100, round(prob * 60  + (20 if any(w in words for w in {"hate","disgusting","horrible"}) else 0), 1))
    threats     = min(100, round(prob * 40  + (40 if any(w in words for w in {"kill","die","hurt"}) else 0), 1))
    insults     = min(100, round(prob * 70  + (15 if any(w in words for w in {"stupid","idiot","dumb","moron"}) else 0), 1))

    session_log.append({
        "text": text[:60] + ("…" if len(text) > 60 else ""),
        "verdict": "Toxic" if toxic else "Safe",
        "confidence": f"{confidence}%",
    })

    return (
        gr.update(visible=False),
        gr.update(value=_verdict_html(toxic, confidence),              visible=True),
        gr.update(value=_flagged_html(flagged),                        visible=True),
        gr.update(value=_breakdown_html(harassment, hate_speech, threats, insults), visible=True),
        gr.update(value=_explanation(toxic, confidence, flagged),      visible=True),
        gr.update(value=_confidence_bar_html(confidence, toxic),       visible=True),
        render_history(),
        render_stats(),
    )

def _verdict_html(toxic, confidence):
    color, bg, icon, label = (
        ("#ef4444","#2d1414","🚨","Toxic / Harmful Content Detected") if toxic
        else ("#22c55e","#122d1a","✅","Safe Content — No Harmful Language")
    )
    return f"""
<div style="background:{bg};border:1.5px solid {color}40;border-radius:12px;padding:16px 20px;display:flex;align-items:center;gap:14px">
  <span style="font-size:24px">{icon}</span>
  <div>
    <div style="color:{color};font-weight:700;font-size:15px">{label}</div>
    <div style="color:{color}99;font-size:12px;margin-top:3px">Confidence: {confidence}%</div>
  </div>
</div>"""

def _confidence_bar_html(confidence, toxic):
    color = "#ef4444" if toxic else "#22c55e"
    return f"""
<div style="padding:4px 0">
  <div style="display:flex;justify-content:space-between;margin-bottom:6px">
    <span style="font-size:12px;color:#9ca3af;font-weight:500;letter-spacing:0.5px">CONFIDENCE SCORE</span>
    <span style="font-size:14px;font-weight:700;color:#f9fafb;font-family:monospace">{confidence}%</span>
  </div>
  <div style="height:8px;background:#1f2937;border-radius:4px;overflow:hidden">
    <div style="height:100%;width:{confidence}%;background:{color};border-radius:4px"></div>
  </div>
</div>"""

def _flagged_html(terms):
    if not terms:
        return "<div style='color:#6b7280;font-size:13px;padding:8px 0'>No specific toxic terms flagged.</div>"
    tags = "".join(
        f'<span style="background:#3b1212;color:#f87171;border:1px solid #7f1d1d44;border-radius:20px;padding:3px 12px;font-size:12px;font-weight:500">{t}</span>'
        for t in terms
    )
    return f"""
<div>
  <div style="font-size:11px;color:#6b7280;letter-spacing:0.6px;margin-bottom:8px;font-weight:500">FLAGGED TERMS</div>
  <div style="display:flex;flex-wrap:wrap;gap:6px">{tags}</div>
</div>"""

def _breakdown_html(harassment, hate_speech, threats, insults):
    rows = [
        ("Harassment",  harassment,  "#a78bfa"),
        ("Hate Speech", hate_speech, "#f472b6"),
        ("Threats",     threats,     "#f87171"),
        ("Insults",     insults,     "#fb923c"),
    ]
    bars = "".join(f"""
<div style="display:flex;align-items:center;gap:10px;margin-bottom:8px">
  <span style="color:#9ca3af;font-size:12px;min-width:90px">{label}</span>
  <div style="flex:1;height:5px;background:#1f2937;border-radius:3px;overflow:hidden">
    <div style="height:100%;width:{val}%;background:{color};border-radius:3px"></div>
  </div>
  <span style="color:#6b7280;font-size:11px;font-family:monospace;min-width:34px;text-align:right">{val}%</span>
</div>""" for label, val, color in rows)
    return f"""
<div>
  <div style="font-size:11px;color:#6b7280;letter-spacing:0.6px;margin-bottom:10px;font-weight:500">RISK BREAKDOWN</div>
  {bars}
</div>"""

def _explanation(toxic, confidence, flagged):
    if toxic:
        term_note = f" Flagged terms: {', '.join(flagged)}." if flagged else ""
        return f"Classified as **toxic** with {confidence}% confidence.{term_note} This comment may contain abusive or harmful language."
    return f"Classified as **safe** with {confidence}% confidence. No harmful language patterns were detected."

def render_history():
    if not session_log:
        return pd.DataFrame(columns=["#", "Comment", "Verdict", "Confidence"])
    df = pd.DataFrame(session_log[::-1])
    df.insert(0, "#", range(len(df), 0, -1))
    df.columns = ["#", "Comment", "Verdict", "Confidence"]
    return df

def render_stats():
    total = len(session_log)
    if total == 0:
        return "<div style='color:#6b7280;font-size:13px;text-align:center;padding:10px'>No data yet</div>"
    toxic_n = sum(1 for r in session_log if r["verdict"] == "Toxic")
    safe_n  = total - toxic_n
    rate    = round(toxic_n / total * 100, 1)
    def card(label, value, color):
        return f"""
<div style="background:#111827;border:0.5px solid #374151;border-radius:10px;padding:14px 16px;text-align:center">
  <div style="font-size:11px;color:#6b7280;letter-spacing:0.7px;margin-bottom:6px">{label}</div>
  <div style="font-size:22px;font-weight:700;font-family:monospace;color:{color}">{value}</div>
</div>"""
    return f"""
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px">
  {card("TOTAL", total, "#a78bfa")}
  {card("TOXIC", toxic_n, "#f87171")}
  {card("SAFE", safe_n, "#4ade80")}
  {card("TOXIC RATE", f"{rate}%", "#fb923c")}
</div>"""

def clear_session():
    session_log.clear()
    return render_history(), render_stats()

# ── UI ────────────────────────────────────────────────────────────────────────
EXAMPLES = [
    "you are stupid and ugly",
    "have a wonderful day, stay safe!",
    "I hate you so much, go away",
    "you are amazing and kind",
    "nobody likes you, just give up",
    "great work on the project today!",
]

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.violet,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
    ).set(
        body_background_fill="#0d0d14",
        block_background_fill="#111827",
        block_border_color="#1f2937",
        input_background_fill="#0a0a10",
        button_primary_background_fill="#5b3fd4",
        button_primary_background_fill_hover="#6d4fe4",
        button_primary_text_color="white",
    ),
    title="CyberShield AI",
) as demo:

    gr.HTML("""
    <div style="padding:24px 0 8px;border-bottom:1px solid #1f2937;margin-bottom:24px">
      <div style="display:flex;align-items:center;gap:12px;margin-bottom:6px">
        <div style="width:38px;height:38px;background:linear-gradient(135deg,#5b3fd4,#a855f7);border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:20px">🛡</div>
        <span style="font-family:monospace;font-size:22px;font-weight:700;color:#f0eeff;letter-spacing:-0.5px">CyberShield AI</span>
      </div>
      <p style="color:#6b7280;font-size:13px;margin:0 0 0 50px;letter-spacing:0.3px">Real-time toxic comment detection · Scikit-learn · TF-IDF · Logistic Regression</p>
    </div>
    """)

    stats_display = gr.HTML(render_stats())
    gr.HTML("<div style='height:16px'></div>")

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            with gr.Group():
                comment_input = gr.Textbox(lines=7, placeholder="Type or paste a comment to analyze...", label="Comment", show_copy_button=True)
                analyze_btn   = gr.Button("🔍  Analyze Comment", variant="primary", size="lg")
            error_box = gr.Markdown(visible=False)
            gr.Examples(examples=EXAMPLES, inputs=comment_input, label="Quick Examples")

        with gr.Column(scale=5):
            verdict_out   = gr.HTML(visible=False)
            conf_out      = gr.HTML(visible=False)
            flagged_out   = gr.HTML(visible=False)
            breakdown_out = gr.HTML(visible=False)
            explain_out   = gr.Markdown(visible=False)

    gr.HTML("<div style='height:24px'></div>")

    with gr.Group():
        with gr.Row():
            gr.HTML("<div style='font-size:11px;color:#6b7280;letter-spacing:0.7px;font-weight:500;padding:4px 0'>ANALYSIS HISTORY</div>")
            clear_btn = gr.Button("Clear", size="sm", variant="secondary")
        history_table = gr.DataFrame(value=render_history(), interactive=False, wrap=True)

    outputs = [error_box, verdict_out, flagged_out, breakdown_out, explain_out, conf_out, history_table, stats_display]
    analyze_btn.click(fn=predict, inputs=comment_input, outputs=outputs)
    comment_input.submit(fn=predict, inputs=comment_input, outputs=outputs)
    clear_btn.click(fn=clear_session, outputs=[history_table, stats_display])

demo.launch(debug=True)

/tmp/ipykernel_2550/2449702286.py:170: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a5dc89e849e14cf78b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
